In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, floor, months_between, get
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.dim_product_price")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "dim_product_price"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_product_price.ipynb"
_silver_table = "silver.product_price"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.dim_product_price


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 3
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------------+--------------------------+--------------------------+--------------------------+
|product_id|store |channel|list_price|cost_price|currency|effective_start_date|is_current|is_deleted|source_file    |created_on                |created_by                |modified_on               |modified_by               |
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------------+--------------------------+--------------------------+--------------------------+
|P1001     |AMZ_US|MKT    |500.00    |350.00    |USD     |2025-01-01          |true      |false     |dim_product.csv|2026-01-29 01:12:21.098955|silver_product_price.ipynb|2026-01-29 01:12:21.098955|silver_product_price.ipynb|
|P1002     |EBAY  |MKT    |1200.00   |900.00    |USD     

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 3


In [10]:
# Generating surrogate key and business key

df_sk = df.withColumn("product_price_sk", xxhash64(concat_ws("||", "product_id", "store", "channel", "effective_start_date")).cast("bigint")) \
          .withColumn("product_price_bk", xxhash64(concat_ws("||", "product_id", "store", "channel")).cast("bigint"))    

df_sk.show(truncate = False)

+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------+--------------------+
|product_id|store |channel|list_price|cost_price|currency|effective_start_date|is_current|is_deleted|source_file    |created_on                |created_by                |modified_on               |modified_by               |product_price_sk    |product_price_bk    |
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------+--------------------+
|P1001     |AMZ_US|MKT    |500.00    |350.00    |USD     |2025-01-01          |true      |false     |dim_product.csv|2026-01-29 01:12:21.098955|silver_product_price.ipynb|2026-01-29 01:12:21.09895

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_gold_currency = spark.read.table("gold.dim_currency").select("currency_sk", "currency_code")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_currency, how = "inner", on=df_sk.currency==df_gold_currency.currency_code) \
                 .select("product_price_sk", "product_price_bk", "product_id", "store_sk", "channel_sk", "currency_sk", "list_price", "cost_price",
                         "effective_start_date", "is_current", "is_deleted", "source_file")
                

df_silver.show(truncate = False)

+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+----------+----------+---------------+
|product_price_sk    |product_price_bk    |product_id|store_sk|channel_sk|currency_sk|list_price|cost_price|effective_start_date|is_current|is_deleted|source_file    |
+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+----------+----------+---------------+
|-6617344052881870074|1826704936074838732 |P1001     |2       |1         |1          |500.00    |350.00    |2025-01-01          |true      |false     |dim_product.csv|
|-2105855115900565320|-1809102942906596247|P1003     |5       |1         |1          |500.00    |350.00    |2025-01-01          |true      |false     |dim_product.csv|
|-9188969364773753618|-3228085214969746345|P1002     |5       |1         |1          |1200.00   |900.00    |2025-02-15          |true      |false     |dim_produ

In [12]:
# Truncate table for full load
dt_dim_product_price = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_dim_product_price.delete()
    dt_dim_product_price.vacuum(0)

#Assuming file can have mixed cases for sending records - could be with history versions or sometimes only latest rows
#SCD2 - Merge 1 for dates already in dim table
dt_dim_product_price.alias("t").merge(
    df_silver.alias("s"), 
    "t.product_price_bk = s.product_price_bk and t.is_current = true and s.effective_start_date > t.effective_start_date"
    ).whenMatchedUpdate(set={
    "effective_end_date" : "date_sub(s.effective_start_date, 1)",
    "is_current" : "false",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()

print("SPARK-APP: Merge1 completed")

SPARK-APP: Merge1 completed


In [13]:
dt_dim_product_price.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows").show(1, False)

+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|0      |null           |null                 |null                |null         |
+-------+---------------+---------------------+--------------------+-------------+



In [14]:

window = Window.partitionBy("product_price_bk").orderBy("effective_start_date")

df_silver_nextDate = df_silver.withColumn("next_date", lead("effective_start_date").over(window)) \
                              .withColumn("effective_end_date", when(col("next_date").isNotNull(), date_sub("next_date",1)).otherwise(lit(None).cast("date"))) \
                              .withColumn("is_current", col("next_date").isNull()) \
                              .drop("next_date")

In [15]:
df_silver_nextDate.show()
df_silver_nextDate.printSchema()

+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+----------+----------+---------------+------------------+
|    product_price_sk|    product_price_bk|product_id|store_sk|channel_sk|currency_sk|list_price|cost_price|effective_start_date|is_current|is_deleted|    source_file|effective_end_date|
+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+----------+----------+---------------+------------------+
|-9188969364773753618|-3228085214969746345|     P1002|       5|         1|          1|   1200.00|    900.00|          2025-02-15|      true|     false|dim_product.csv|              null|
|-2105855115900565320|-1809102942906596247|     P1003|       5|         1|          1|    500.00|    350.00|          2025-01-01|      true|     false|dim_product.csv|              null|
|-6617344052881870074| 1826704936074838732|     P1001|       2|  

In [16]:
# Merge 2 for new dates not in dim table
dt_dim_product_price.alias("t").merge(
    df_silver_nextDate.alias("s"),
    "t.product_price_bk = s.product_price_bk and s.effective_start_date = t.effective_start_date"
).whenNotMatchedInsert(values = {
    "product_price_sk": "s.product_price_sk", 
    "product_price_bk": "s.product_price_bk",
    "product_id": "s.product_id",
    "store_sk"   : "s.store_sk",
    "channel_sk" : "s.channel_sk",
    "currency_sk" : "s.currency_sk",
    "list_price": "s.list_price",
    "cost_price": "s.cost_price",
    "effective_start_date" : "s.effective_start_date",
    "effective_end_date" : "s.effective_end_date",
    "is_current"  : "s.is_current",
    "is_deleted"  : "s.is_deleted",
    "source_file" : "s.source_file",
    "created_on"  : "current_timestamp()",
    "created_by"  : f"'{_script_name}'",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()
                     
print("SPARK-APP: Merge2 completed")    

SPARK-APP: Merge2 completed


In [17]:
hist = dt_dim_product_price.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           4718|                    3|                   0|            3|
+-------+---------------+---------------------+--------------------+-------------+



In [18]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [19]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+-----------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name|       table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+-----------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|dim_product_price|delta-load|     merge|2026-01-29 02:57:...|    success|           3|2026-01-29 02:57:...|gold_product_pric...|2026-01-29 02:57:...|gold_product_pric...|
+-----+-----------+-----------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [20]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+
|    product_price_sk|    product_price_bk|product_id|store_sk|channel_sk|currency_sk|list_price|cost_price|effective_start_date|effective_end_date|is_current|is_deleted|          created_on|          created_by|         modified_on|         modified_by|    source_file|
+--------------------+--------------------+----------+--------+----------+-----------+----------+----------+--------------------+------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+
|-9188969364773753618|-3228085214969746345|     P1002|       5|         1|          1|   1200.00|    900.00|          2025-02-15|              null|      true|     false|2026-01-29 02:57:

In [21]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [22]:
spark.stop()